# Mini-TP 3 — Sirve tu modelo por gRPC

Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual

Expone por gRPC el mismo modelo que la Sesión 1 sirve por REST: XGBoost que predice si un
crimen reportado en Chicago (2024) termina en arresto.

## Qué se entrega

1. `tp3_grpc/scoring.proto` con mensajes de entrada y salida tipados.
2. Los stubs generados y un servidor que carga el modelo una sola vez.
3. Un cliente que llama al servicio (unary).
4. Un método de server-streaming que puntúa un lote.
5. La comparación de latencia contra el endpoint REST, con la reflexión al final.

## Cómo está organizado

El código vive en los `.py` del repo (`tp3_grpc/server.py`, `client.py`, `benchmark.py`) y está
cubierto por tests; este notebook los importa, los muestra y los ejecuta. De esa forma no hay dos
copias del mismo código que se desincronicen.

In [1]:
import logging
import os
from pathlib import Path

import arrest_model

# El notebook vive en tp3_grpc/, pero protoc y los imports esperan la raíz del repo.
REPO = Path(arrest_model.__file__).resolve().parent.parent
os.chdir(REPO)

# Los dos servicios dejan una línea de log por request, y acá corren en este mismo proceso: esas
# líneas irían a la salida de las celdas. Se silencian LOS DOS por dos razones: para no llenar
# el notebook de ruido, y porque durante el benchmark el costo de escribirlas se le cobraría al
# protocolo que quedara sin silenciar, y la comparación dejaría de ser justa.
for servicio in ("tp1_rest", "tp3_grpc"):
    logging.getLogger(servicio).setLevel(logging.WARNING)

print("Trabajando desde:", REPO)

Trabajando desde: /Users/belcattaneo/Documents/MIA/MLOPS2/mlops2-minitps


## 1. El modelo

Un único `model/model.pkl` con el `XGBClassifier` entrenado y los parámetros de codificación.
Lo comparten los tres mini-TPs: gRPC no reimplementa nada del modelo, solo lo expone.

In [2]:
from arrest_model.model import load_bundle

bundle = load_bundle()
metadata = bundle["metadata"]
print(f"{metadata['name']} v{metadata['version']} ({metadata['framework']})")
print("entradas crudas:", metadata["inputs"])
print("features del modelo:", metadata["features"])
print("MCC en test:", round(metadata["metrics"]["mcc"], 4))

chicago-arrest-xgboost v1 (xgboost 3.4.1)
entradas crudas: ['iucr', 'primary_type', 'location_description', 'date', 'latitude', 'longitude']
features del modelo: ['IUCR_freq', 'Primary_Type_freq', 'Location_Description_freq', 'Day_sin', 'X Coordinate_standardized', 'Y Coordinate_standardized', 'Distance Crime To Police Station_standardized']
MCC en test: 0.5811


## 2. El contrato `.proto`

El contrato declara campos nombrados y tipados, no un vector anónimo de números: son los mismos
6 campos crudos que valida la API REST. `date` viaja como `google.protobuf.Timestamp`, es decir
un instante absoluto; el servidor lo convierte a hora de Chicago, que es como se entrenó la
feature `Day_sin`.

In [3]:
from IPython.display import Code, display

display(Code(filename="tp3_grpc/scoring.proto"))

// Contrato del servicio de scoring por gRPC.
//
// Tipa los mismos 6 campos crudos que el contrato REST (arrest_model.schemas.CrimeReport):
// el servidor aplica la misma validación y la misma codificación antes de puntuar.
// Los stubs se regeneran con `make grpc-stubs`; los *_pb2*.py no se editan a mano.

syntax = "proto3";

package arrest;

import "google/protobuf/timestamp.proto";

// Reporte crudo de un crimen en Chicago.
message CrimeReport {
  string iucr = 1;                          // 3 dígitos y un dígito o letra, ej. "1310"
  string primary_type = 2;                  // uno de los 31 tipos vistos en train
  optional string location_description = 3; // si falta, se codifica como UNKNOWN
  google.protobuf.Timestamp date = 4;       // instante UTC; el servidor lo pasa a hora de Chicago
  optional double latitude = 5;             // optional para distinguir "falta" de "vale 0"
  optional double longitude = 6;
}

// Lote de reportes para puntuar en una sola llamada.
message CrimeBatch {
  repeated CrimeReport items = 1;
}

// Predicción y versión del modelo que la produjo.
message Prediction {
  int32 arrest = 1;
  double probability = 2;
  string model_name = 3;
  int32 model_version = 4;
}

service ArrestScoring {
  // Unary: un reporte, una predicción.
  rpc Predict(CrimeReport) returns (Prediction);

  // Server streaming: un lote, N predicciones en el mismo orden que los items.
  rpc PredictStream(CrimeBatch) returns (stream Prediction);
}

## 3. Genera los stubs

`protoc` lee el `.proto` y genera dos archivos: `scoring_pb2.py` con las clases de mensajes y
`scoring_pb2_grpc.py` con el stub del cliente y la clase base del servidor. No se editan a mano:
se regeneran con `make grpc-stubs`, y un test compara los versionados contra el `.proto` para
detectar que alguien lo haya cambiado sin regenerar.

In [4]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "grpc_tools.protoc", "-I", ".",
     "--python_out=.", "--grpc_python_out=.", "tp3_grpc/scoring.proto"],
    check=True,
)
print("generados:", sorted(p.name for p in Path("tp3_grpc").glob("scoring_pb2*.py")))

generados: ['scoring_pb2.py', 'scoring_pb2_grpc.py']


## 4. El servidor

El modelo se carga una sola vez al arrancar (en `serve`) y queda en el servicer; cada RPC solo
valida y puntúa. La validación reusa el `CrimeReport` de la API REST, así que un payload inválido
da el mismo error en los dos protocolos: acá como `INVALID_ARGUMENT`.

In [5]:
display(Code(filename="tp3_grpc/server.py", language="python"))

"""Mini-TP 3: servidor gRPC que sirve el modelo de arrestos de Chicago.

Mismo núcleo que la API REST: valida con `CrimeReport` y puntúa con `predict`. Lo único
propio de acá es traducir entre los mensajes protobuf y ese núcleo.
"""

import logging
import time
from collections.abc import Callable, Iterator, Sequence
from concurrent import futures
from datetime import UTC
from typing import Any

import grpc
from pydantic import ValidationError

from arrest_model.model import load_bundle, predict
from arrest_model.schemas import CrimeReport, PredictionOut
from tp3_grpc import scoring_pb2, scoring_pb2_grpc

logger = logging.getLogger("tp3_grpc")

# Acá no hay uvicorn que configure el logging: sin esto los INFO se descartan en el contenedor.
logging.basicConfig(level=logging.INFO, format="%(levelname)s:     %(message)s")


def report_from_proto(message: scoring_pb2.CrimeReport) -> dict[str, Any]:
    """Pasa el mensaje a un dict con **solo los campos presentes**.

    Lo ausente se omite en vez de viajar en cero: así un campo que falta da error de
    validación y no una predicción sobre un dato inventado.
    """
    report: dict[str, Any] = {"iucr": message.iucr, "primary_type": message.primary_type}
    if message.HasField("location_description"):
        report["location_description"] = message.location_description
    if message.HasField("date"):
        # ToDatetime() devuelve UTC sin zona; se la ponemos para que CrimeReport la convierta
        # a hora de Chicago, que es como se entrenó el Day_sin.
        report["date"] = message.date.ToDatetime().replace(tzinfo=UTC)
    for field in ("latitude", "longitude"):
        if message.HasField(field):
            report[field] = getattr(message, field)
    return report


def validation_detail(error: ValidationError) -> str:
    """Resume los errores nombrando el campo que falló, sin exponer nada interno."""
    return "; ".join(
        f"{'.'.join(str(part) for part in item['loc'])}: {item['msg']}" for item in error.errors()
    )


def prediction_to_proto(prediction: PredictionOut) -> scoring_pb2.Prediction:
    """Pasa la predicción del núcleo al mensaje del contrato gRPC."""
    return scoring_pb2.Prediction(
        arrest=prediction.arrest,
        probability=prediction.probability,
        model_name=prediction.model_name,
        model_version=prediction.model_version,
    )


def validated_reports(
    messages: Sequence[scoring_pb2.CrimeReport], context: grpc.ServicerContext, *, indexed: bool
) -> list[CrimeReport]:
    """Valida **todos** los mensajes antes de que se puntúe ninguno.

    Con `indexed`, el detalle del error dice qué item del lote falló: en un stream, si no
    se validara todo primero, el cliente recibiría predicciones y recién después el error.
    """
    reports = []
    for index, message in enumerate(messages):
        try:
            reports.append(CrimeReport.model_validate(report_from_proto(message)))
        except ValidationError as exc:
            prefix = f"item {index}: " if indexed else ""
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, prefix + validation_detail(exc))
    return reports


class ArrestScoringServicer(scoring_pb2_grpc.ArrestScoringServicer):
    """Sirve un modelo ya cargado; la carga es responsabilidad de `serve`."""

    def __init__(self, bundle: dict[str, Any]) -> None:
        self.bundle = bundle

    def Predict(  # noqa: N802 (el nombre lo fija el .proto)
        self, request: scoring_pb2.CrimeReport, context: grpc.ServicerContext
    ) -> scoring_pb2.Prediction:
        """Unary: valida un reporte y devuelve su predicción."""
        reports = validated_reports([request], context, indexed=False)
        return prediction_to_proto(predict(self.bundle, reports)[0])

    def PredictStream(  # noqa: N802 (el nombre lo fija el .proto)
        self, request: scoring_pb2.CrimeBatch, context: grpc.ServicerContext
    ) -> Iterator[scoring_pb2.Prediction]:
        """Server streaming: valida el lote enter

In [6]:
from tp3_grpc.server import serve

# port=0 -> el sistema elige uno libre. serve() arranca y devuelve sin bloquear.
servidor, puerto = serve(port=0, bundle=bundle)
print("servidor gRPC escuchando en el puerto", puerto)

servidor gRPC escuchando en el puerto 58591


## 5. El cliente (unary)

`stub.Predict(...)` parece una llamada local, pero serializa el mensaje, lo manda por HTTP/2 y
deserializa la respuesta. Eso es el *remote procedure call*.

In [7]:
import grpc

from tp3_grpc import scoring_pb2, scoring_pb2_grpc
from tp3_grpc.client import crime_report

canal = grpc.insecure_channel(f"127.0.0.1:{puerto}")
stub = scoring_pb2_grpc.ArrestScoringStub(canal)

prediccion = stub.Predict(crime_report())
print("arrest:", prediccion.arrest, "| probability:", prediccion.probability)
print("modelo:", prediccion.model_name, "v" + str(prediccion.model_version))

arrest: 0 | probability: 0.06843266636133194
modelo: chicago-arrest-xgboost v1


In [8]:
# Un reporte inválido no llega al modelo: el contrato lo corta antes.
try:
    stub.Predict(crime_report(primary_type="BANANA"))
except grpc.RpcError as error:
    print(error.code().name, "->", error.details()[:90], "...")

INVALID_ARGUMENT -> primary_type: Input should be 'ARSON', 'ASSAULT', 'BATTERY', 'BURGLARY', 'CONCEALED CARRY  ...


## 6. Streaming

`PredictStream` recibe un lote y devuelve N respuestas, una por item. El servidor valida todo el
lote antes de emitir nada (si no, el cliente recibiría predicciones y recién después el error) y
llama al modelo una sola vez para las N filas.

In [9]:
lote = scoring_pb2.CrimeBatch(
    items=[crime_report(), crime_report(primary_type="THEFT")] * 5
)
for i, p in enumerate(stub.PredictStream(lote), start=1):
    print(f"[{i:2}] arrest={p.arrest} probability={p.probability:.6f}")

[ 1] arrest=0 probability=0.068433
[ 2] arrest=1 probability=0.765073
[ 3] arrest=0 probability=0.068433
[ 4] arrest=1 probability=0.765073
[ 5] arrest=0 probability=0.068433
[ 6] arrest=1 probability=0.765073
[ 7] arrest=0 probability=0.068433
[ 8] arrest=1 probability=0.765073
[ 9] arrest=0 probability=0.068433
[10] arrest=1 probability=0.765073


In [10]:
import time

# ¿Las predicciones llegan de a una o todas juntas al final?
grande = scoring_pb2.CrimeBatch(items=[crime_report()] * 100)
inicio = time.perf_counter()
primera = None
for i, _ in enumerate(stub.PredictStream(grande)):
    if i == 0:
        primera = (time.perf_counter() - inicio) * 1000
ultima = (time.perf_counter() - inicio) * 1000
print(f"lote de 100 -> primera a los {primera:.2f} ms, última a los {ultima:.2f} ms")

lote de 100 -> primera a los 3.25 ms, última a los 7.07 ms


## 7. Comparación de latencia gRPC vs REST

La medición no se corre dentro de este notebook, por dos razones de método:

1. La mitad de la comparación son las cohortes dockerizadas, y un notebook no puede meterse a sí
   mismo en un contenedor. Esa parte tenía que medirse afuera sí o sí; medir la otra mitad acá
   adentro daría dos procedimientos distintos para una sola comparación.
2. Con los dos servicios corriendo como hilos de este mismo proceso, el trabajo que hacen por
   request compite con el cliente que mide. Probado: así el ratio da alrededor de 1.1x, contra
   1.4x–1.6x cuando los servicios corren como procesos aparte, que es lo que pasa en un
   despliegue real.

La medición vive entonces en un procedimiento repetible y fuera de acá: se levantan los dos
servicios (los dos locales, o los dos en contenedor) y se corre `make grpc-bench`, que usa
`tp3_grpc/benchmark.py`. Los resultados están en la sección siguiente.

In [11]:
display(Code(filename="tp3_grpc/benchmark.py", language="python"))

"""Compara la latencia de gRPC contra REST sirviendo el mismo modelo.

Simetría: los dos protocolos hacen exactamente el mismo trabajo (validar, codificar y predecir),
los dos dejan una línea de log por request, y se miden en el mismo lugar. Si uno corriera en
contenedor y el otro local, se estaría midiendo el empaquetado y no el protocolo.

Se mide REST de dos formas a propósito: sin sesión (conexión nueva por llamada) y con keep-alive.
gRPC reutiliza el canal siempre, así que la comparación justa es contra keep-alive; la otra
muestra cuánto cuesta el handshake.

Toda respuesta REST se valida con `raise_for_status()`: sin eso, un servicio que contesta 503 se
mediría como si fueran predicciones y daría a REST una ventaja falsa.

Uso: `uv run python -m tp3_grpc.benchmark [--n 300] [--rest-url ...] [--grpc-target ...]`
"""

import argparse
import math
import time
from collections.abc import Callable
from typing import Any

import grpc
import requests

from arrest_model.schemas import EXAMPLE_REPORT
from tp3_grpc import scoring_pb2, scoring_pb2_grpc
from tp3_grpc.client import crime_report

REST_URL = "http://127.0.0.1:8000"
GRPC_TARGET = "127.0.0.1:50051"


def percentile(samples: list[float], fraction: float) -> float:
    """Percentil por rango más cercano: el p95 de 100 muestras es la muestra 95."""
    if not samples:
        raise ValueError("no hay muestras para calcular el percentil")
    rank = max(1, math.ceil(fraction * len(samples)))
    return sorted(samples)[rank - 1]


def summarize(samples: list[float]) -> dict[str, float]:
    """Media, mediana y p95 de las latencias, en ms."""
    if not samples:
        raise ValueError("no hay muestras para resumir")
    return {
        "media": sum(samples) / len(samples),
        "p50": percentile(samples, 0.50),
        "p95": percentile(samples, 0.95),
    }


def elapsed_ms(call: Callable[[], Any]) -> float:
    """Milisegundos que tarda una sola llamada."""
    started = time.perf_counter()
    call()
    return (time.perf_counter() - started) * 1000


def measure(call: Callable[[], Any], n: int, warmup: int) -> dict[str, float]:
    """Descarta `warmup` llamadas y resume la latencia de las `n` siguientes."""
    if n < 1:
        raise ValueError("hacen falta al menos 1 repetición para medir")
    for _ in range(warmup):
        call()
    return summarize([elapsed_ms(call) for _ in range(n)])


def run_benchmark(
    rest_url: str = REST_URL,
    grpc_target: str = GRPC_TARGET,
    n: int = 300,
    warmup: int = 30,
    batch_size: int = 100,
) -> dict[str, Any]:
    """Corre las dos comparaciones (una llamada, y un lote) y devuelve los números."""
    if batch_size < 1:
        raise ValueError("el lote tiene que tener al menos 1 reporte")
    payload = dict(EXAMPLE_REPORT)
    predict_url, batch_url = f"{rest_url}/v1/predict", f"{rest_url}/v1/predict/batch"
    message = crime_report()
    batch = scoring_pb2.CrimeBatch(items=[crime_report()] * batch_size)

    def post(caller: Any, url: str, body: dict[str, Any], timeout: int = 10) -> Any:
        """POST que falla fuerte si la respuesta no es 2xx, para no medir errores."""
        response = caller.post(url, json=body, timeout=timeout)
        response.raise_for_status()
        return response

    with requests.Session() as session, grpc.insecure_channel(grpc_target) as channel:
        grpc.channel_ready_future(channel).result(timeout=15)
        stub = scoring_pb2_grpc.ArrestScoringStub(channel)
        unary = {
            "REST sin sesión": measure(lambda: post(requests, predict_url, payload), n, warmup),
            "REST keep-alive": measure(lambda: post(session, predict_url, payload), n, warmup),
            "gRPC": measure(lambda: stub.Predict(message), n, warmup),
        }
        lote = {
            f"{batch_size} POST /v1/predict": elapsed_ms(
                lambda: [post(session, predict_url, payload) for _ in range(batch_size)]
            ),
            "1 POST /v1/predict/batch": elapsed_ms(


Para reproducir la medición, desde la raíz del repo:

```bash
# cohortes rest-local y grpc-local
uv run uvicorn tp1_rest.app:app --port 8000 &
uv run python -m tp3_grpc.server &
make grpc-bench

# cohortes rest-dockerizado y grpc-dockerizado
make rest-up && make grpc-up
make grpc-bench
```

## 8. Reflexión

> Pendiente de completar. Abajo están los datos medidos; las conclusiones se escriben después de
> analizarlos.

### Cómo se midió

Para que la comparación mida el protocolo y no otra cosa, los dos lados tienen que hacer el mismo
trabajo y medirse en el mismo lugar:

- Los dos servicios dejan una línea de log por request. REST la tiene desde el TP1; al servidor
  gRPC se le agregó un interceptor justamente para emparejar, porque si no REST pagaba un costo
  que gRPC no y la diferencia medida lo habría incluido.
- Toda respuesta REST se valida con `raise_for_status()`. Sin eso, un servicio que contesta 503
  se mide como si fueran predicciones: probado a propósito, daba REST cinco veces más rápido que
  gRPC, o sea la conclusión al revés.
- REST se mide sin sesión y con keep-alive, porque gRPC siempre reutiliza el canal. La
  comparación justa es contra keep-alive; la otra muestra cuánto cuesta abrir la conexión.

Cohortes: `rest-local` y `grpc-local` son los dos servicios como proceso en la máquina;
`rest-dockerizado` y `grpc-dockerizado`, los dos en contenedor al mismo tiempo. Nunca se compara
una cohorte local contra una dockerizada de otro protocolo.

Tres corridas por condición, 300 repeticiones cada una, todas seguidas en una misma sesión. Los
valores absolutos se mueven con el estado de la máquina, así que solo son comparables entre sí
las corridas de una misma tanda.

### La tabla

Cada bloque es una cohorte; el nombre se escribe una sola vez y las filas siguientes son del
mismo bloque.

| cohorte | prueba | variante | corridas (ms) | mediana |
|---|---|---|---|---|
| `rest-local` | una llamada | sin sesión | 2.78 · 2.82 · 2.76 | 2.78 |
|  | una llamada | keep-alive | 2.82 · 2.64 · 2.66 | 2.66 |
|  | lote de 100 | 100 llamadas sueltas | 275.61 · 252.94 · 255.83 | 255.83 |
|  | lote de 100 | 1 llamada al batch | 5.86 · 3.32 · 3.07 | 3.32 |
| `rest-dockerizado` | una llamada | sin sesión | 5.58 · 6.08 · 5.63 | 5.63 |
|  | una llamada | keep-alive | 4.94 · 5.34 · 4.83 | 4.94 |
|  | lote de 100 | 100 llamadas sueltas | 491.85 · 527.44 · 507.65 | 507.65 |
|  | lote de 100 | 1 llamada al batch | 6.92 · 7.60 · 6.08 | 6.92 |
| `grpc-local` | una llamada | canal reusado | 1.81 · 1.89 · 1.85 | 1.85 |
|  | lote de 100 | 1 PredictStream | 8.82 · 6.42 · 6.46 | 6.46 |
| `grpc-dockerizado` | una llamada | canal reusado | 2.81 · 2.81 · 2.75 | 2.81 |
|  | lote de 100 | 1 PredictStream | 12.41 · 13.46 · 16.78 | 13.46 |

Dos datos medidos aparte: el mismo reporte pesa 60 bytes en protobuf contra 161 en JSON (2.7x),
porque en protobuf viajan los números de campo y no sus nombres; y la celda de más arriba muestra
que en un `PredictStream` de 100 la primera predicción llega bastante antes que la última.

La corrida del benchmark que se ve unas celdas más arriba es una medición local independiente,
que se rehace cada vez que se ejecuta el notebook.

### Qué se repite

Rangos separados, o sea efecto reproducible:

- gRPC contra REST keep-alive: 1.40x–1.56x local y 1.76x–1.90x dockerizado. La ventaja de gRPC
  se agranda dentro de Docker.
- Dockerizar cuesta caro y no por igual: la mediana empeora +102% en REST sin sesión, +86% en
  REST keep-alive y +52% en gRPC. gRPC aguanta bastante mejor, pero está lejos de ser inmune.
- Agrupar es lo que más mueve la aguja: 100 llamadas sueltas contra una sola al batch da entre
  47x y 83x, en las dos condiciones.
- `POST /v1/predict/batch` le gana a `PredictStream` en las seis corridas.

### Qué no se puede concluir

Que la ventaja del batch sobre el stream dependa de dockerizar: da 1.51x–2.11x local contra
1.77x–2.76x dockerizado, o sea rangos superpuestos. La dirección se repite siempre; la magnitud
no se distingue del ruido.

Los números varían entre corridas (es una laptop con Docker Desktop, no un banco de pruebas):
sirven para comparar rangos, no para la tercera cifra.

### A responder

La consigna pide 3–5 líneas:

1. ¿Qué diferencia de latencia observé, y a qué la atribuyo?
2. ¿Cuándo usaría gRPC y cuándo REST en esta plataforma?
3. ¿Qué costo tiene gRPC (contrato, tooling, no lo habla el navegador directo)?

## 9. Limpieza

In [12]:
canal.close()
servidor.stop(0)
print("servidor gRPC detenido")

servidor gRPC detenido
